# STAT/CS 2870 Final Project

Maria, Alexa, Wai

In [10]:
# Alexa's Data Cleaning

#Getting rid of incident number repeats
stops_unique = stops.drop_duplicates(subset=['incident_number'])
#Making call_time a date variable
stops_unique['call_time'] = pd.to_datetime(stops_unique['call_time'])
#Making missing variables in Race Null
stops_unique = stops_unique.replace('Missing', np.nan)
#Getting rid of impossible and missing age values and replacing them with null
stops_unique.loc[stops_unique['age'] > 100, 'age'] = np.nan
stops_unique.loc[stops_unique['age'] < 10, 'age'] = np.nan
stops_unique = stops_unique.replace('NA', np.nan)
stops_unique['age'].describe()
#Getting rid of the prefix 'M' in stop_type
stops_unique['stop_type'] = stops_unique['stop_type'].str[4:]
stops_unique.head()


# Wai's Data Cleaning
#read in data: 
import pandas as pd
import numpy as np

df = pd.read_csv('Traffic_Stops.csv')
#After inspecting the unique categories in each of the columns (column 12 - 25) with .unique(), 
#we decided to recode the search_type 

search_type_map = {
    "NS = No search": "no_search",
    "SPC = Consent Search w/PC": "consent_search",
    "SW = Search w/ Warrant": "warrant_search",
    "SRS = Search Reasonable Suspicion":"resonable_suspicion_search",
#    "not recorded": np.nan
}

df["search_type_clean"] = df["search_type"].replace(search_type_map)


display((df).head())


# Maria's Data Cleaning

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

if 'stops' not in globals():
    stops = pd.read_csv('Traffic_Stops.csv', encoding='utf-8-sig', low_memory=False)

stops = stops.copy()
cols = ['stop_outcome', 'veh_state', 'count_tickets', 'count_warnings', 'search', 'search_type']
print('Starting rows:', len(stops))
print(stops[cols].isna().sum())
for col in cols:
    print('\n', col)
    print(stops[col].value_counts(dropna=False).head(25))

# Clean stop outcomes
stops_raw = stops.copy()

def strip_prefix(series):
    return (series.astype('string').str.strip()
            .str.replace(r'^[^=]+\s*=\s*', '', regex=True).str.strip())

outcome = strip_prefix(stops['stop_outcome']).str.lower()
stops['stop_outcome_clean'] = outcome.map({'ticket': 'Ticket', 'warning': 'Warning'})
print('Unrecognized or excluded outcome labels:')
print(pd.crosstab(stops['stop_outcome'].fillna('<NA>'), stops['stop_outcome_clean'].fillna('Excluded')))
rows_before = len(stops)
stops = stops.loc[stops['stop_outcome_clean'].notna()].copy()
print(f'Kept {len(stops):,} of {rows_before:,} rows; excluded {rows_before-len(stops):,}.')


# Clean state plate
state = stops['veh_state'].astype('string').str.strip().str.upper()
state = state.replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, 'NULL': pd.NA,
                       'MISSING': pd.NA, 'UNKNOWN': pd.NA, 'VERMONT': 'VT'})
stops['veh_state_clean'] = state
# Only valid two-letter US state/DC abbreviations count as known states.
valid_states = set('AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC'.split())
stops['veh_state_group'] = 'Missing/other'
stops.loc[state == 'VT', 'veh_state_group'] = 'Vermont'
stops.loc[state.isin(valid_states) & (state != 'VT'), 'veh_state_group'] = 'Out-of-state'
print(stops['veh_state_group'].value_counts(dropna=False))
print('Nonstandard values (review before deciding how to group):')
print(state.loc[state.notna() & ~state.isin(valid_states)].value_counts().head(30))

# Clean ticket & warning count
for col in ['count_tickets', 'count_warnings']:
    numeric = pd.to_numeric(stops[col], errors='coerce')
    invalid = numeric.notna() & ((numeric < 0) | (numeric % 1 != 0))
    print(col, '| missing/unreadable:', numeric.isna().sum(), '| negative/noninteger:', invalid.sum())
    stops[col + '_clean'] = numeric.mask(invalid).astype('Int64')
    print(stops[col + '_clean'].value_counts(dropna=False).head(15))

print('Ticket outcomes with zero/missing ticket count:',
      ((stops['stop_outcome_clean'] == 'Ticket') &
       (stops['count_tickets_clean'].fillna(0) == 0)).sum())
print('Warning outcomes with zero/missing warning count:',
      ((stops['stop_outcome_clean'] == 'Warning') &
       (stops['count_warnings_clean'].fillna(0) == 0)).sum())

# Clean search & search type
flag = stops['search'].astype('string').str.strip().str.lower()
stops['search_clean'] = flag.map({'1': 1, '0': 0, 'true': 1, 'false': 0,
                                  'yes': 1, 'no': 0, '1.0': 1, '0.0': 0}).astype('Int64')
print('Search flag values not recognized:')
print(stops.loc[stops['search_clean'].isna(), 'search'].value_counts(dropna=False))

type_clean = strip_prefix(stops['search_type']).str.lower()
type_clean = type_clean.replace({'': pd.NA, 'missing': pd.NA, 'unknown': pd.NA,
                                 'nan': pd.NA, 'none': 'no search'})
stops['search_type_clean'] = type_clean.str.title()
print('Clean search types:')
print(stops['search_type_clean'].value_counts(dropna=False))
print('Search flag vs. search type:')
print(pd.crosstab(stops['search_clean'].fillna(-1),
                  stops['search_type_clean'].fillna('Missing'), dropna=False))
conflicts = (stops['search_clean'].eq(1).fillna(False) &
             stops['search_type_clean'].eq('No Search').fillna(False)) | (
             stops['search_clean'].eq(0).fillna(False) &
             stops['search_type_clean'].notna() &
             stops['search_type_clean'].ne('No Search').fillna(False))
print('Potential contradictions:', conflicts.sum())

,incident_number,call_type,call_time,race,gender,age,stop_type,stop_outcome,veh_state,count_tickets,...,viol_ls,viol_miss,viol_other,viol_vehicle,viol_civil,viol_muni,arrest,arrest_warrant,ObjectId,search_type_clean
0,12BU000766,Traffic,2012/01/09 15:03:42+00,White,Male - M,52.0,O = Other,T = Ticket,VT,1,...,0,1,0,0,0,0,none,NaN,1,no_search
1,12BU000767,DLS,2012/01/09 15:07:16+00,White,Male - M,41.0,M = Moving violation,A = Arrest for Violation,VT,2,...,0,1,0,0,0,0,arrest,False,2,no_search
2,12BU000772,Traffic,2012/01/09 15:27:17+00,White,Male - M,32.0,V = Vehicle Equipment,T = Ticket,VT,1,...,0,1,0,0,0,0,none,NaN,3,no_search
3,12BU000785,Traffic,2012/01/09 17:41:23+00,White,Female - F,21.0,V = Vehicle Equipment,W = Warning,MA,0,...,0,0,0,1,0,0,none,NaN,4,no_search
4,12BU000788,Traffic,2012/01/09 18:08:34+00,White,Male - M,19.0,V = Vehicle Equipment,W = Warning,VT,0,...,0,0,0,1,0,0,none,NaN,5,no_search


Starting rows: 35847
stop_outcome        0
veh_state           0
count_tickets       0
count_warnings      0
search              0
search_type       182
dtype: int64

 stop_outcome
stop_outcome
W = Warning    27432
T = Ticket      8415
Name: count, dtype: int64

 veh_state
veh_state
VT      30237
MA       1040
NY        928
NONE      589
CT        485
NH        389
NJ        379
PA        222
ME        216
FL        157
PQ        152
MD        103
RI         95
VA         84
CA         67
NC         55
MI         50
OH         49
CO         45
IL         45
GA         39
TX         37
TN         34
SC         32
ON         27
Name: count, dtype: int64

 count_tickets
count_tickets
0    27721
1     7257
2      722
3      111
4       27
5        5
6        2
8        1
7        1
Name: count, dtype: int64

 count_warnings
count_warnings
1    28567
0     6598
2      605
3       70
4        6
5        1
Name: count, dtype: int64

 search
search
0    35357
1      490
Name: count, dtype: int